# ENTSO-E data exploration

Explore the ingested ENTSO-E data from Bronze through Gold. The notebook profiles coverage, resolutions, revisions, price distributions, and price time series by bidding zone and auction series. It is read-only.

Set the widgets, then run all cells. The default price chart uses the full available date range; use the start and end widgets to narrow it. The end date is exclusive and interpreted as UTC.


In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import pandas as pd

dbutils.widgets.text("catalog", "eu_electricity_forecast", "Catalog")
dbutils.widgets.text("bronze_schema", "bronze", "Bronze schema")
dbutils.widgets.text("silver_schema", "silver", "Silver schema")
dbutils.widgets.text("gold_schema", "gold", "Gold schema")
dbutils.widgets.dropdown("zone", "ALL", ["ALL", "DE_LU", "FR", "NL", "LT"], "Price chart zone")
dbutils.widgets.text("start_date", "2024-10-01", "Start date UTC (inclusive)")
dbutils.widgets.text("end_date", "", "End date UTC (exclusive; blank = latest)")

catalog = dbutils.widgets.get("catalog").strip()
bronze_schema = dbutils.widgets.get("bronze_schema").strip()
silver_schema = dbutils.widgets.get("silver_schema").strip()
gold_schema = dbutils.widgets.get("gold_schema").strip()
zone_filter = dbutils.widgets.get("zone")
start_date = dbutils.widgets.get("start_date").strip()
end_date = dbutils.widgets.get("end_date").strip()

def table_name(schema, name):
    def quoted(value):
        return "`" + value.replace("`", "``") + "`"
    return f"{quoted(catalog)}.{quoted(schema)}.{quoted(name)}"

def table_exists(schema, name):
    return spark.catalog.tableExists(table_name(schema, name))

bronze_name = table_name(bronze_schema, "bronze_entsoe_observations")
price_history_name = table_name(silver_schema, "silver_prices")
price_current_name = table_name(silver_schema, "silver_prices_current")


## Table inventory

Counts are exact Spark counts and scan each listed table. `_current` objects are views over their versioned Silver tables.


In [ ]:
candidates = [
    ("Bronze", bronze_schema, "bronze_entsoe_observations", "Append-only source observations"),
    ("Silver", silver_schema, "silver_prices", "Price revisions"),
    ("Silver", silver_schema, "silver_prices_current", "Latest price revision view"),
    ("Silver", silver_schema, "silver_load", "Load observations"),
    ("Silver", silver_schema, "silver_generation", "Generation observations"),
    ("Silver", silver_schema, "silver_forecasts", "Forecast observations"),
    ("Silver", silver_schema, "silver_cross_border_flows", "Directed flow observations"),
    ("Silver", silver_schema, "silver_quarantine", "Rejected or unmapped observations"),
    ("Gold", gold_schema, "day_ahead_features", "Point-in-time model inputs"),
    ("Gold", gold_schema, "day_ahead_price_labels", "Latest known target prices"),
    ("Gold", gold_schema, "day_ahead_training_data", "Features joined to labels"),
]

inventory = []
for layer, schema, name, purpose in candidates:
    if table_exists(schema, name):
        df = spark.table(table_name(schema, name))
        inventory.append({
            "layer": layer,
            "object": f"{catalog}.{schema}.{name}",
            "type": "view" if name.endswith("_current") or name.endswith("_data") else "table",
            "rows": df.count(),
            "columns": len(df.columns),
            "purpose": purpose,
        })

if inventory:
    display(spark.createDataFrame(inventory).orderBy("layer", "object"))
else:
    print("No expected tables found. Check the catalog and schema widgets.")


## Silver coverage and resolution

Compare the available time span and row counts across data products and zones. Missing intervals are absent records, not zero values.


In [ ]:
silver_tables = [
    "silver_prices",
    "silver_load",
    "silver_generation",
    "silver_forecasts",
    "silver_cross_border_flows",
]
coverage = []
for name in silver_tables:
    if table_exists(silver_schema, name):
        source = spark.table(table_name(silver_schema, name))
        coverage.append(
            source.groupBy("data_item", "bidding_zone").agg(
                F.count("*").alias("rows"),
                F.countDistinct("timestamp_utc").alias("distinct_intervals"),
                F.min("timestamp_utc").alias("first_interval_utc"),
                F.max("timestamp_utc").alias("last_interval_utc"),
                F.countDistinct("resolution_minutes").alias("resolution_count"),
            )
        )

if coverage:
    coverage_df = coverage[0]
    for part in coverage[1:]:
        coverage_df = coverage_df.unionByName(part)
    display(coverage_df.orderBy("data_item", "bidding_zone"))
else:
    print("Silver tables are not available yet.")

if table_exists(silver_schema, "silver_prices"):
    display(
        spark.table(price_history_name)
        .where(F.col("data_item") == "day_ahead_prices")
        .groupBy("bidding_zone", "resolution_minutes")
        .agg(F.count("*").alias("price_versions"), F.min("timestamp_utc").alias("first_utc"), F.max("timestamp_utc").alias("last_utc"))
        .orderBy("bidding_zone", "resolution_minutes")
    )


## Price revisions and auction series

The base price table preserves source versions. Its `_current` view selects the latest version for each logical observation. Auction series are kept separate because a zone can have more than one price series.


In [ ]:
if table_exists(silver_schema, "silver_prices"):
    price_history = spark.table(price_history_name).where(
        (F.col("data_item") == "day_ahead_prices") & (F.col("unit") == "EUR/MWh")
    )
    version_counts = (
        price_history.groupBy("bidding_zone", "source_record_key")
        .agg(F.countDistinct("source_record_version_key").alias("versions"))
    )
    display(
        version_counts.groupBy("bidding_zone")
        .agg(
            F.count("*").alias("logical_observations"),
            F.sum(F.when(F.col("versions") > 1, 1).otherwise(0)).alias("observations_with_revisions"),
            F.max("versions").alias("max_versions_for_one_observation"),
        )
        .orderBy("bidding_zone")
    )

    price_current = spark.table(price_current_name).where(
        (F.col("data_item") == "day_ahead_prices") & (F.col("unit") == "EUR/MWh")
    )
    auction_fields = [name for name in ["market_agreement_id", "auction_type", "auction_category", "business_type"] if name in price_current.columns]
    display(
        price_current.groupBy("bidding_zone", *auction_fields)
        .agg(
            F.count("*").alias("current_price_rows"),
            F.min("timestamp_utc").alias("first_utc"),
            F.max("timestamp_utc").alias("last_utc"),
        )
        .orderBy("bidding_zone", *auction_fields)
    )
else:
    print("Silver price table is not available yet.")


## Price distribution and outliers

Prices can be negative and can include spikes. Review the quantiles and largest absolute values before selecting model filters or transformations.


In [ ]:
if table_exists(silver_schema, "silver_prices_current"):
    price_stats = price_current.groupBy("bidding_zone").agg(
        F.count("*").alias("rows"),
        F.min("value").alias("min_eur_mwh"),
        F.expr("percentile_approx(value, 0.01, 10000)").alias("p01_eur_mwh"),
        F.expr("percentile_approx(value, 0.50, 10000)").alias("median_eur_mwh"),
        F.expr("percentile_approx(value, 0.99, 10000)").alias("p99_eur_mwh"),
        F.max("value").alias("max_eur_mwh"),
        F.sum(F.when(F.col("value") < 0, 1).otherwise(0)).alias("negative_price_rows"),
        F.sum(F.when(F.abs(F.col("value")) >= 1000, 1).otherwise(0)).alias("abs_price_at_least_1000_rows"),
    )
    display(price_stats.orderBy("bidding_zone"))

    display(
        price_current.select(
            "bidding_zone", "timestamp_utc", "interval_end_utc", "resolution_minutes",
            "value", "source_revision", "market_agreement_id", "auction_type",
        )
        .orderBy(F.abs(F.col("value")).desc())
        .limit(25)
    )


## Price time series

The first chart shows the exact interval observations for the selected date range. Each line is a zone and auction series; overlapping lines can occur when series share values. For long ranges, Databricks' display chart tools can zoom and filter the results.


In [ ]:
if table_exists(silver_schema, "silver_prices_current"):
    plot_prices = price_current
    if zone_filter != "ALL":
        plot_prices = plot_prices.where(F.col("bidding_zone") == zone_filter)
    if start_date:
        plot_prices = plot_prices.where(F.col("timestamp_utc") >= F.to_timestamp(F.lit(start_date)))
    if end_date:
        plot_prices = plot_prices.where(F.col("timestamp_utc") < F.to_timestamp(F.lit(end_date)))

    series_cols = [name for name in ["market_agreement_id", "auction_type", "auction_category"] if name in plot_prices.columns]
    chart_cols = ["timestamp_utc", "bidding_zone", "value", *series_cols]
    plot_spark = plot_prices.select(*chart_cols).orderBy("timestamp_utc")
    display(plot_spark)

    plot_pdf = plot_spark.toPandas()
    if not plot_pdf.empty:
        plot_pdf["timestamp_utc"] = pd.to_datetime(plot_pdf["timestamp_utc"], utc=True)
        if series_cols:
            plot_pdf["auction_series"] = plot_pdf[series_cols].fillna("").astype(str).agg(" / ".join, axis=1).str.strip(" /")
            plot_pdf["auction_series"] = plot_pdf["auction_series"].replace("", "unspecified series")
        else:
            plot_pdf["auction_series"] = "price series"
        plot_pdf["line"] = plot_pdf["bidding_zone"] + " | " + plot_pdf["auction_series"]

        detail_start = plot_pdf["timestamp_utc"].max() - pd.Timedelta(weeks=4)
        detail_pdf = plot_pdf[plot_pdf["timestamp_utc"] >= detail_start]
        fig, (overview_ax, detail_ax) = plt.subplots(2, 1, figsize=(16, 10), sharey=True)
        for label, group in plot_pdf.groupby("line", sort=True):
            overview_ax.plot(group["timestamp_utc"], group["value"], linewidth=0.7, alpha=0.75, label=label)
            recent = detail_pdf[detail_pdf["line"] == label]
            detail_ax.plot(recent["timestamp_utc"], recent["value"], linewidth=0.9, alpha=0.8, label=label)
        overview_ax.set_title("ENTSO-E day-ahead prices — selected date range")
        overview_ax.set_ylabel("EUR/MWh")
        overview_ax.grid(True, alpha=0.25)
        overview_ax.legend(loc="best", fontsize="small")
        detail_ax.set_title("High-resolution view — last four weeks in the selected range")
        detail_ax.set_xlabel("Delivery interval (UTC)")
        detail_ax.set_ylabel("EUR/MWh")
        detail_ax.grid(True, alpha=0.25)
        detail_ax.legend(loc="best", fontsize="small")
        fig.tight_layout()
        plt.show()
    else:
        print("No price observations match the selected zone and dates.")


## Gold features and labels

Gold labels hold the latest known price; the feature table preserves inputs as known at each forecast cutoff. Never include `label_price_eur_mwh` or label metadata among model predictors.


In [ ]:
training_name = table_name(gold_schema, "day_ahead_training_data")
labels_name = table_name(gold_schema, "day_ahead_price_labels")
features_name = table_name(gold_schema, "day_ahead_features")

if spark.catalog.tableExists(training_name):
    training = spark.table(training_name)
    summary = training.groupBy("bidding_zone").agg(
        F.count("*").alias("labelled_feature_rows"),
        F.min("target_interval_start_utc").alias("first_target_utc"),
        F.max("target_interval_start_utc").alias("last_target_utc"),
        F.expr("percentile_approx(label_price_eur_mwh, 0.50, 10000)").alias("median_label_eur_mwh"),
        F.sum(F.when(F.col("price_lag_24h_eur_mwh_present"), 1).otherwise(0)).alias("rows_with_24h_lag"),
        F.sum(F.when(F.col("price_lag_168h_eur_mwh_present"), 1).otherwise(0)).alias("rows_with_168h_lag"),
    )
    display(summary.orderBy("bidding_zone"))
    display(training.orderBy(F.desc("forecast_issue_time_utc")).limit(10))
else:
    print("Gold training view is not available yet. Run the ingestion job after deploying the latest bundle.")


## Reading the results

- Check zone and product coverage before comparing model performance across regions.
- Inspect resolution changes and gaps; do not fill missing prices with zero.
- Use the current price view for the latest target label, and the versioned table to study corrections.
- Historical price lags use the latest Silver values and assume they were known at the forecast cutoff. Their first-seen timestamps are backfill download times, not original publication times; rare later corrections could therefore appear in older examples.
- The provisional 10:00 Brussels issue cutoff and the observed auction series still need validation before treating the training view as production-ready.
